# 05 — Exportar ResNet-50 para AWS

**Etapa 1 del proyecto AWS.** Genera los 3 archivos que viajan a la nube:

1. `resnet50.pt` — pesos del modelo (~100 MB)
2. `clases.json` — 38 nombres de carpeta en orden
3. `nombres_display.json` — etiquetas legibles en español

**No exportes** el dataset de 87k imágenes ni los DataLoaders.

Ejecutar en Lightning Studio después de tener `checkpoints/resnet50.pt`.

## 1. Rutas (ajusta si tu carpeta difiere)

In [ ]:
from pathlib import Path
import sys

# Raíz del repo (sube dos niveles desde notebooks/)
PROJECT_ROOT = Path("/teamspace/studios/this_studio/plant-disease-detector-aws")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Checkpoint del notebook 03-resnet50
CHECKPOINT = PROJECT_ROOT / "checkpoints" / "resnet50.pt"

# Symlinks limpios del compañero / tu EDA
TRAIN_DIR = Path("/teamspace/studios/this_studio/proyecto-plantas/data/plantas_train")
if not TRAIN_DIR.exists():
    TRAIN_DIR = PROJECT_ROOT / "data" / "plantas_train"

OUT_DIR = PROJECT_ROOT / "artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Imagen suelta para probar inferencia
SAMPLE_IMAGE = None  # ej: TRAIN_DIR / "Tomato___healthy" / "alguna.jpg"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHECKPOINT:", CHECKPOINT, "→ existe:", CHECKPOINT.is_file())
print("TRAIN_DIR:", TRAIN_DIR, "→ existe:", TRAIN_DIR.is_dir())
print("OUT_DIR:", OUT_DIR)

## 2. Lista de clases (mismo orden que ImageFolder)

In [ ]:
from src.class_mapping import classes_from_train_dir, save_class_artifacts

class_names = classes_from_train_dir(TRAIN_DIR)
print(f"Clases encontradas: {len(class_names)}")
print("Primeras 5:", class_names[:5])

## 3. Exportar JSON + copiar checkpoint

In [ ]:
import shutil

clases_path, nombres_path = save_class_artifacts(class_names, OUT_DIR)

dest_pt = OUT_DIR / "resnet50.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"No está el checkpoint. Entrena primero o corrige CHECKPOINT: {CHECKPOINT}"
    )
shutil.copy2(CHECKPOINT, dest_pt)

print("✓ Archivos listos para AWS:")
print(" ", dest_pt, f"({dest_pt.stat().st_size / 1e6:.1f} MB)")
print(" ", clases_path)
print(" ", nombres_path)

## 4. Cargar modelo y verificar pesos

In [ ]:
import torch
from src.model_resnet50 import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_checkpoint(str(dest_pt), num_classes=len(class_names), device=device)
print("Modelo cargado en", device)
print("Parámetros (millones):", sum(p.numel() for p in model.parameters()) / 1e6)

## 5. Probar inferencia en una imagen

In [ ]:
import json
from src.inference import predict_image

display_names = json.loads(nombres_path.read_text(encoding="utf-8"))

if SAMPLE_IMAGE is None:
    # Toma la primera imagen que encuentre en train
    for folder in sorted(TRAIN_DIR.iterdir()):
        if folder.is_dir():
            imgs = list(folder.glob("*.jpg")) + list(folder.glob("*.JPG")) + list(folder.glob("*.png"))
            if imgs:
                SAMPLE_IMAGE = imgs[0]
                break

from pathlib import Path

if SAMPLE_IMAGE is None or not Path(SAMPLE_IMAGE).is_file():
    raise FileNotFoundError("Define SAMPLE_IMAGE o verifica TRAIN_DIR")

result = predict_image(model, SAMPLE_IMAGE, class_names, display_names, device=device)
print("Imagen:", SAMPLE_IMAGE)
print("Diagnóstico:", result["predictions"][0]["display_name"])
print("Confianza:", f"{result['confidence']*100:.2f}%")
print("Top 3:")
for p in result["predictions"]:
    print(f"  - {p['display_name']}: {p['confidence']*100:.2f}%")

## 6. Siguiente paso

1. Descarga la carpeta `artifacts/` desde Lightning.
2. Crea el **Budget alert** en AWS (ver `infra/GUIA_AWS.md`).
3. Sube los 3 archivos a S3 (Etapa 2).

El repo paralelo en GitHub: `plant-disease-detector-aws` (no toca el proyecto EfficientNet/ONNX).